In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import norm

from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim


# Inference-Time Temperature Rescaling (TSR) Figures

This notebook visualizes inference-time temperature control for 1D DDPM sampling. We use pre-trained checkpoints directly (assumed to already be available) and compare original, flattened, and sharpened outputs.

## 1) Setup and Configuration

Define device, diffusion length, template distributions, and checkpoint directory used throughout the notebook.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_diffusion_steps = 100

datasets = {
    "single": {"dataset_shape": (50_000, 1), "means": [0.0], "stds": 1.0},
    "barrier": {"dataset_shape": (50_000, 1), "means": [-3.0, 3.0], "stds": 0.5},
    "composed": {
        "dataset_shape": (100_000, 1),
        "means": [-3.0, 0.0, 3.0],
        "stds": [0.5, 1.0, 0.5],
    },
}

ckpt_dir = "model_checkpoints"

## 2) Template Distribution Utilities

These utilities generate mixture samples and closed-form density curves so we can compare empirical histograms with known targets.

These helpers (1) sample from the configured Gaussian mixture and (2) compute the corresponding analytic PDF. Each component is weighted equally, and `k` controls flattening/sharpening through variance rescaling.

In [ ]:
@torch.no_grad()
def generate_gaussian_mixture(dataset_name, device='cpu'):
	"""Generates mixture of gaussians according to inputted means and standard deviations"""

	dataset_config = datasets[dataset_name]
	dataset_shape = dataset_config["dataset_shape"]
	n_samples = dataset_shape[0]

	means = torch.as_tensor(dataset_config['means'], dtype=torch.float32)
	stds = dataset_config['stds']
	
	n_gaussians = len(means)
	
	if isinstance(stds, (int, float)):
		stds = torch.full((n_gaussians,), float(stds))
	else:
		stds = torch.as_tensor(stds, dtype=torch.float32)
		assert len(stds) == n_gaussians, f"stds length {len(stds)} != n_gaussians {n_gaussians}"
		
	component_ids = np.random.choice(n_gaussians, size=n_samples)
	samples = torch.zeros(n_samples, 1, device=device)
	
	for i in range(n_gaussians):
		mask = component_ids == i
		samples[mask] = torch.normal(
			mean=float(means[i]),
			std=float(stds[i]),
			size=(mask.sum(), 1)
		).to(device)
	
	return samples


@torch.no_grad()
def compute_mixture_pdf(dataset_name, x_axis, k=1.0):
	"""Computes analytical pdf of training dataset from dataset config file, used for plotting"""

	dataset_config = datasets[dataset_name]
	means = np.array(dataset_config['means'])
	stds = dataset_config['stds']
		
	if isinstance(stds, (int, float)):
		stds = np.full(len(means), stds)
	else:
		stds = np.array(stds)
		
	stds = stds / np.sqrt(k)
	
	n_gaussians = len(means)
	pdf = np.zeros_like(x_axis)
	
	for mu, sigma in zip(means, stds):
		pdf += norm.pdf(x_axis, loc=mu, scale=sigma)
	
	pdf /= n_gaussians  
	
	return pdf 

Quick sanity-check plot for any template distribution using sampled histogram vs analytic PDF.

In [ ]:
def plot_samples(dataset_name, x_limit=10, n_bins=200):
    x_axis = np.linspace(-x_limit, x_limit, n_bins)
    bins = np.linspace(-x_limit, x_limit, n_bins)

    samples = generate_gaussian_mixture(dataset_name, device=device).cpu().numpy().reshape(-1)
    pdf = compute_mixture_pdf(dataset_name, x_axis)

    plt.figure(figsize=(7, 4))
    plt.hist(samples, bins=bins, density=True, alpha=0.45, label="Samples")
    plt.plot(x_axis, pdf, linewidth=2.0, label="Analytic PDF")
    plt.title(f"Template distribution: {dataset_name}")
    plt.xlabel("x")
    plt.ylabel("density")
    plt.legend()
    plt.tight_layout()

## 3) Noise Schedule and TSR Coefficients

We use a cosine DDPM schedule and precompute the standard scalar terms used during reverse-time updates.

In [ ]:
@torch.no_grad()
def cosine_beta_schedule(timesteps, s=0.008):
	"""Cosine noise schedule, taken from reduce reuse recycle code"""
	steps = timesteps + 1
	t = torch.linspace(0, timesteps, steps, dtype=torch.float32) / timesteps
	alphas_cumprod = torch.cos((t + s) / (1 + s) * math.pi * 0.5) ** 2
	alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
	betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
	return torch.clip(betas, 0, 0.999)

betas = cosine_beta_schedule(n_diffusion_steps).to(device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)  # alphā_t

ts_desc = torch.arange(n_diffusion_steps - 1, -1, -1, device=device)

`compute_tsr_schedule` applies temporal score rescaling (TSR) at each diffusion step for a requested temperature factor `k`.

In [ ]:
@torch.no_grad()
def compute_tsr_schedule(k, sigma, t):
	"""Computes temporal score rescaling coefficient. Outpit will be shape (N_DIFFUSION_STEPS, n_langevin_steps)"""

	a_bar = alpha_bars[t]
	sigma_t = torch.sqrt(1.0 - a_bar)
	alpha_t = torch.sqrt(a_bar)

	eta_t = (alpha_t**2) / (sigma_t**2)
	num = eta_t * (sigma ** 2) + 1
	den = (eta_t * (sigma ** 2)) / k + 1
	tsr = num / den

	return tsr

## 4) DDPM Sampling Core

This section keeps the reverse DDPM update logic intact and applies TSR-adjusted score estimates during sampling.

In [ ]:
@torch.no_grad()
def compute_score(model, x, t, k, sigma):
	"""Computes score = - epsilon * temp / √(1 - α_bar)"""
	
	x_shape = x.shape
	ones = torch.ones((x_shape[0], 1), device=device)
	eps_hat = model(x, t * ones)   
	a_bar = alpha_bars[t]
	temp_t = compute_tsr_schedule(k, sigma, t)
	score_hat= - eps_hat * temp_t / torch.sqrt(1.0 - a_bar)
	return score_hat

In [ ]:
@torch.no_grad()
def ddpm_tsr(model, dataset_shape, k=1.0, sigma=1.0):
	"""Sampling algorithm for DDPM, ULA, and MALA"""

	x = torch.randn(dataset_shape, device=device)
		
	for t in ts_desc: 

		alpha_t = alphas[t]
		beta_t = betas[t]
		sqrt_alpha_t = torch.sqrt(alpha_t)
		sqrt_beta_t = torch.sqrt(beta_t)
		noise = torch.randn(dataset_shape, device=device)

		score_hat = compute_score(model, x, t, k, sigma)
		x = (x + beta_t * score_hat) / sqrt_alpha_t + sqrt_beta_t * noise

	return x

## 5) Model Definition and Loading

Define the denoiser architecture and a small checkpoint loader utility for inference-only use.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
	def __init__(self, dim):
		super().__init__()
		self.dim = dim

	def forward(self, t):
		if t.dim() == 2:
			t = t.squeeze(-1)

		half = self.dim // 2
		freqs = torch.exp(
			-math.log(10000) * torch.arange(half, device=t.device) / half
		)
		args = t[:, None] * freqs[None, :]
		emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

		if self.dim % 2 == 1:
			emb = F.pad(emb, (0, 1))

		return emb


class MLP(nn.Module):
    def __init__(
        self,
        x_dim=1,
        hidden_dim=512,   # 128 -> 512
        time_dim=64,      # 32 -> 64
        n_layers=8,       # 4 -> 8
    ):
        super().__init__()

        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.input = nn.Linear(x_dim + time_dim, hidden_dim)
        self.layers = nn.ModuleList(
            [nn.Linear(hidden_dim, hidden_dim) for _ in range(n_layers)]
        )
        self.output = nn.Linear(hidden_dim, x_dim)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        h = torch.cat([x, t_emb], dim=-1)
        h = F.silu(self.input(h))
        for layer in self.layers:
            h = h + F.silu(layer(h))
        return self.output(h)


In [ ]:
def load_model(path):
	"""Load trained model from checkpoint"""
	model = MLP().to(device)
	model.load_state_dict(torch.load(path, map_location=device))
	model.eval()
	return model

## 6) Standalone Training Code

While models are pretrained, you can alter datasets to retrain models. Function is not called.

In [ ]:
TRAINING = {
    "n_steps": 500_000,
    "batch_size": 512,
    "lr": 2e-4
}

n_steps = TRAINING["n_steps"]
batch_size = TRAINING["batch_size"]
lr = TRAINING["lr"]

def train_model(name, dataset_config, existing_checkpoint=None, load_file=None, k=1.0, log_every=5000):
	"""Trains model according to a dataset defined by dataset_config"""

	x0_all = generate_gaussian_mixture(
        n_samples=dataset_config["dataset_shape"][0],
		means=dataset_config["means"],
		stds=dataset_config["stds"],
        device='cpu',
    )

	loader = DataLoader(
		TensorDataset(x0_all),
		batch_size=batch_size,
		shuffle=True,
		drop_last=True,
		num_workers=0,      # keep simple; set >0 if you want
		pin_memory=True,    # helps H2D transfer
	)

	model = MLP().to(device)
	if existing_checkpoint is not None:
		print(f"Loading {existing_checkpoint}")
		checkpoint = torch.load(existing_checkpoint, map_location=device)
		model.load_state_dict(checkpoint)
		model.eval()

	opt = optim.Adam(model.parameters(), lr=lr)
	model.train()

	data_iter = iter(loader)

	for step in range(1, n_steps + 1):
		try:
			(x0,) = next(data_iter)
		except StopIteration:
			data_iter = iter(loader)  # reshuffles because shuffle=True
			(x0,) = next(data_iter)

		x0 = x0.to(device, non_blocking=True)

		t = torch.randint(0, n_diffusion_steps, (batch_size, 1), device=device)
		a_bar = alpha_bars[t]
		
		noise = torch.randn_like(x0)
		xt = torch.sqrt(a_bar) * x0 + torch.sqrt(1.0 - a_bar) * noise

		xt = xt.detach().requires_grad_(True)  # we need grads wrt xt
		eps_hat = model(xt, t)   # energy values from EBM: shape [B, 1] or [B]

		loss = ((noise - eps_hat) ** 2).mean()

		opt.zero_grad()
		loss.backward()
		opt.step()

		if step % log_every == 0:
			print(f"temperature={k} step={step} loss={loss.item():.4f}")

	save_path = f"{ckpt_dir}/{name}_{k:.1f}.pt"
	torch.save(model.state_dict(), save_path)
	print(f"Trained model saved to {save_path}")

	return model

## 7) Temperature Comparison Figures

Create a three-panel visualization on a non-single template distribution: original (`k=1.0`), flattened (`k=0.5`), and sharpened (`k=2.0`).

In [ ]:
def plot_temperature_triptych(
    dataset_name="composed",
    sigma=1.0,
    x_limit=8,
    n_bins=220,
):
    """Create side-by-side visuals for original, flattened, and sharpened sampling."""

    if dataset_name == "single":
        raise ValueError("Use 'barrier' or 'composed' for template comparison visuals.")

    dataset_shape = datasets[dataset_name]["dataset_shape"]
    model = load_model(f"{ckpt_dir}/{dataset_name}_1.0.pt")

    x_axis = np.linspace(-x_limit, x_limit, n_bins)
    bins = np.linspace(-x_limit, x_limit, n_bins)

    ks = [1.0, 0.5, 2.0]
    titles = ["Original (k = 1.0)", "Flattened (k = 0.5)", "Sharpened (k = 2.0)"]

    fig, axes = plt.subplots(1, 3, figsize=(17, 4), sharey=True)

    for ax, k, title in zip(axes, ks, titles):
        samples = ddpm_tsr(model, dataset_shape, k=k, sigma=sigma)
        samples_np = samples.detach().cpu().numpy().reshape(-1)
        pdf = compute_mixture_pdf(dataset_name, x_axis, k=k)

        ax.hist(samples_np, bins=bins, density=True, alpha=0.45, label="Samples")
        ax.plot(x_axis, pdf, linewidth=2.0, label="Analytic target")
        ax.set_title(title)
        ax.set_xlabel("x")
        ax.grid(alpha=0.2)

    axes[0].set_ylabel("density")
    axes[-1].legend(loc="upper right")
    fig.suptitle(f"TSR temperature comparison on '{dataset_name}' template", y=1.03)
    plt.tight_layout()

    return fig, axes


# Example: non-single template distribution
plot_temperature_triptych(dataset_name="composed", sigma=1.0)